In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

print("📥 Chargement Financial PhraseBank...\n")

# DÉTECTION AUTOMATIQUE DU RÉPERTOIRE DE TRAVAIL
current_dir = Path.cwd()
print(f"📂 Répertoire actuel: {current_dir}")
print(f"   Nom du dossier: {current_dir.name}\n")

# Déterminer la racine du projet
if current_dir.name == 'notebooks':
    # Si on est dans notebooks/, remonter d'un niveau
    project_root = current_dir.parent
    print("✅ Exécution depuis notebooks/")
elif (current_dir / 'notebooks').exists():
    # Si on est déjà à la racine
    project_root = current_dir
    print("✅ Exécution depuis la racine du projet")
else:
    # Chercher la racine en remontant
    temp_dir = current_dir
    while temp_dir.parent != temp_dir:
        if (temp_dir / 'data').exists() and (temp_dir / 'notebooks').exists():
            project_root = temp_dir
            break
        temp_dir = temp_dir.parent
    else:
        project_root = current_dir
    print(f"✅ Racine détectée: {project_root.name}")

# Chemins absolus
data_dir = project_root / "data" / "external"
print(f"\n📁 Dossier data/external:")
print(f"   {data_dir}")
print(f"   Existe? {data_dir.exists()}")

if not data_dir.exists():
    raise FileNotFoundError(f"Le dossier {data_dir} n'existe pas!")

# OPTION 1: Dataset académique (recommandé)
academic_path = data_dir / "FinancialPhraseBank" / "Sentences_AllAgree.txt"

# OPTION 2: Dataset Kaggle
csv_path = data_dir / "all-data.csv"

print(f"\n🔍 Recherche des fichiers:")
print(f"   Academic: {academic_path.exists()} - {academic_path}")
print(f"   Kaggle:   {csv_path.exists()} - {csv_path}")

# CHARGEMENT
df = None

# Essayer dataset académique d'abord
if academic_path.exists():
    print(f"\n✅ CHARGEMENT DU DATASET ACADÉMIQUE")
    print(f"   Source: Malo et al. (2014) - Financial PhraseBank")
    print(f"   Fichier: {academic_path.name}")
    
    try:
        # Format: "sentence@sentiment"
        df = pd.read_csv(
            academic_path,
            sep='@',
            header=None,
            names=['sentence', 'sentiment'],
            encoding='latin-1',
            on_bad_lines='skip'  # Ignorer lignes mal formées
        )
        
        print(f"   ✅ Chargé: {len(df)} lignes brutes")
        
        # Nettoyer les labels
        df['sentiment'] = df['sentiment'].str.strip().str.lower()
        
    except Exception as e:
        print(f"   ❌ Erreur lecture academic: {e}")
        df = None

# Si échec, essayer CSV Kaggle
if df is None and csv_path.exists():
    print(f"\n✅ CHARGEMENT DU DATASET KAGGLE")
    print(f"   Fichier: {csv_path.name}")
    
    try:
        df = pd.read_csv(csv_path, encoding='utf-8')
        
        print(f"   ✅ Chargé: {len(df)} lignes")
        print(f"   Colonnes: {df.columns.tolist()}")
        
        # Normaliser colonnes
        column_mapping = {}
        for col in df.columns:
            col_lower = col.lower()
            if 'sentence' in col_lower or 'text' in col_lower:
                column_mapping[col] = 'sentence'
            elif 'sentiment' in col_lower or 'label' in col_lower:
                column_mapping[col] = 'sentiment'
        
        if column_mapping:
            df.rename(columns=column_mapping, inplace=True)
        
        # Si colonnes numérotées
        if 'sentence' not in df.columns and len(df.columns) >= 2:
            df.columns = ['sentence', 'sentiment'] + list(df.columns[2:])
        
        # Convertir labels numériques
        if df['sentiment'].dtype in ['int64', 'float64']:
            label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
            df['sentiment'] = df['sentiment'].map(label_map)
        
        df['sentiment'] = df['sentiment'].str.strip().str.lower()
        
    except Exception as e:
        print(f"   ❌ Erreur lecture CSV: {e}")
        df = None

# Si tout échoue
if df is None or len(df) == 0:
    print("\n❌ ÉCHEC DU CHARGEMENT")
    print(f"\n📁 Fichiers présents dans {data_dir}:")
    for item in data_dir.iterdir():
        print(f"   {'📁' if item.is_dir() else '📄'} {item.name}")
    
    raise FileNotFoundError("Impossible de charger le dataset!")

# NETTOYAGE
print(f"\n🧹 NETTOYAGE DES DONNÉES")
print(f"{'='*60}")

initial_len = len(df)

# Vérifier colonnes requises
if 'sentence' not in df.columns or 'sentiment' not in df.columns:
    print(f"⚠️ Colonnes trouvées: {df.columns.tolist()}")
    raise ValueError("Colonnes 'sentence' et 'sentiment' manquantes!")

# Nettoyer phrases
df['sentence'] = df['sentence'].astype(str).str.strip()
df['sentence'] = df['sentence'].str.replace(r'\s+', ' ', regex=True)

# Filtrer sentiments valides
valid_sentiments = ['negative', 'neutral', 'positive']
df = df[df['sentiment'].isin(valid_sentiments)].copy()

# Supprimer NaN et duplicats
df = df.dropna(subset=['sentence', 'sentiment'])
df = df[df['sentence'].str.len() >= 10]  # Min 10 caractères
df = df.drop_duplicates(subset=['sentence'])

# Reset index
df = df.reset_index(drop=True)

print(f"   Lignes initiales:  {initial_len:,}")
print(f"   Après nettoyage:   {len(df):,}")
print(f"   Supprimées:        {initial_len - len(df):,}")

# STATISTIQUES
print(f"\n📊 STATISTIQUES FINALES")
print(f"{'='*60}")
print(f"   Total phrases:     {len(df):,}")
print(f"   Longueur moyenne:  {df['sentence'].str.len().mean():.1f} caractères")
print(f"   Longueur min:      {df['sentence'].str.len().min()}")
print(f"   Longueur max:      {df['sentence'].str.len().max()}")

# Distribution
print(f"\n📈 DISTRIBUTION DES SENTIMENTS:")
sentiment_counts = df['sentiment'].value_counts()
for sent in ['positive', 'neutral', 'negative']:
    if sent in sentiment_counts.index:
        count = sentiment_counts[sent]
        pct = count / len(df) * 100
        bar = '█' * int(pct / 2)
        print(f"   {sent:8s}: {count:4d} ({pct:5.1f}%) {bar}")

# Ratio déséquilibre
imbalance_ratio = sentiment_counts.max() / sentiment_counts.min()
print(f"\n⚖️  Ratio déséquilibre: {imbalance_ratio:.2f}x")
if imbalance_ratio > 5:
    print(f"   ⚠️ Classes déséquilibrées")
else:
    print(f"   ✅ Distribution acceptable")

# Sauvegarder
output_path = data_dir / "financial_phrasebank_full.csv"
df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\n💾 Sauvegardé: {output_path.name}")

# Exemples
print(f"\n📝 EXEMPLES PAR SENTIMENT:")
print(f"{'='*60}\n")

for sentiment in ['positive', 'neutral', 'negative']:
    if sentiment in df['sentiment'].values:
        examples = df[df['sentiment'] == sentiment].head(2)
        print(f"{sentiment.upper()}:")
        for _, row in examples.iterrows():
            text = row['sentence']
            if len(text) > 100:
                text = text[:97] + "..."
            print(f"   • {text}")
        print()

print(f"{'='*60}")
print(f"✅ DATASET PRÊT POUR L'AUGMENTATION!")
print(f"{'='*60}")


📥 Chargement Financial PhraseBank...

📂 Répertoire actuel: c:\Users\lenovo\OneDrive\Bureau\cours\sentiTrade-HMA-V2
   Nom du dossier: sentiTrade-HMA-V2

✅ Exécution depuis la racine du projet

📁 Dossier data/external:
   c:\Users\lenovo\OneDrive\Bureau\cours\sentiTrade-HMA-V2\data\external
   Existe? True

🔍 Recherche des fichiers:
   Academic: True - c:\Users\lenovo\OneDrive\Bureau\cours\sentiTrade-HMA-V2\data\external\FinancialPhraseBank\Sentences_AllAgree.txt
   Kaggle:   True - c:\Users\lenovo\OneDrive\Bureau\cours\sentiTrade-HMA-V2\data\external\all-data.csv

✅ CHARGEMENT DU DATASET ACADÉMIQUE
   Source: Malo et al. (2014) - Financial PhraseBank
   Fichier: Sentences_AllAgree.txt
   ✅ Chargé: 2264 lignes brutes

🧹 NETTOYAGE DES DONNÉES
   Lignes initiales:  2,264
   Après nettoyage:   2,258
   Supprimées:        6

📊 STATISTIQUES FINALES
   Total phrases:     2,258
   Longueur moyenne:  121.9 caractères
   Longueur min:      11
   Longueur max:      315

📈 DISTRIBUTION DES SENTIME

In [ ]:
import random
import re
from typing import List
from tqdm.auto import tqdm

print("🔄 AUGMENTATION DES DONNÉES")
print(f"{'='*60}\n")

def augment_financial_sentence(sentence: str, sentiment: str, n_variants: int = 2) -> List[str]:
    """
    Générer des variantes d'une phrase financière
    
    Méthodes:
    1. Substitution de synonymes financiers
    2. Paraphrases légères
    3. Réorganisation syntaxique mineure
    """
    
    # Dictionnaires de synonymes par contexte financier
    financial_synonyms = {
        # Verbes de croissance (positifs)
        'increased': ['rose', 'grew', 'climbed', 'jumped', 'surged', 'gained', 'advanced'],
        'grew': ['increased', 'rose', 'expanded', 'climbed', 'advanced'],
        'rose': ['increased', 'grew', 'climbed', 'advanced', 'gained'],
        'surged': ['jumped', 'soared', 'spiked', 'rocketed'],
        'improved': ['strengthened', 'enhanced', 'advanced', 'progressed'],
        
        # Verbes de déclin (négatifs)
        'decreased': ['declined', 'fell', 'dropped', 'slipped', 'weakened', 'tumbled'],
        'fell': ['declined', 'decreased', 'dropped', 'slumped', 'slipped'],
        'declined': ['decreased', 'fell', 'dropped', 'weakened'],
        'dropped': ['fell', 'declined', 'slipped', 'tumbled'],
        'weakened': ['declined', 'deteriorated', 'softened'],
        
        # Noms financiers
        'profit': ['earnings', 'income', 'gains', 'returns'],
        'earnings': ['profit', 'income', 'gains'],
        'revenue': ['sales', 'turnover', 'income'],
        'sales': ['revenue', 'turnover'],
        'loss': ['deficit', 'shortfall'],
        
        # Adjectifs positifs
        'strong': ['robust', 'solid', 'healthy', 'firm'],
        'good': ['positive', 'favorable', 'solid'],
        'positive': ['favorable', 'encouraging', 'good'],
        
        # Adjectifs négatifs
        'weak': ['poor', 'soft', 'lackluster', 'disappointing'],
        'poor': ['weak', 'disappointing', 'lackluster'],
        'negative': ['unfavorable', 'adverse', 'disappointing'],
        
        # Verbes neutres
        'reported': ['announced', 'disclosed', 'stated', 'revealed'],
        'announced': ['reported', 'disclosed', 'stated'],
        'said': ['stated', 'announced', 'noted', 'indicated'],
        
        # Noms d'entreprises
        'company': ['firm', 'corporation', 'group', 'business'],
        'firm': ['company', 'corporation', 'business'],
    }
    
    # Patterns de transformation
    number_patterns = {
        'EUR': ['EUR', '€'],
        'USD': ['USD', '$'],
        '%': ['%', 'percent', 'pct'],
    }
    
    augmented = [sentence]  # Toujours inclure l'originale
    
    # Générer n_variants variantes
    attempts = 0
    max_attempts = n_variants * 3  # Éviter boucle infinie
    
    while len(augmented) < n_variants + 1 and attempts < max_attempts:
        attempts += 1
        new_sentence = sentence.lower()
        modified = False
        
        # Méthode 1: Substitution de synonymes (60% du temps)
        if random.random() < 0.6:
            # Chercher un mot à remplacer
            words = new_sentence.split()
            for i, word in enumerate(words):
                # Nettoyer ponctuation
                clean_word = re.sub(r'[^\w\s]', '', word)
                
                if clean_word in financial_synonyms:
                    synonyms = financial_synonyms[clean_word]
                    if synonyms:
                        replacement = random.choice(synonyms)
                        # Remplacer avec ponctuation préservée
                        pattern = re.compile(r'\b' + re.escape(clean_word) + r'\b', re.IGNORECASE)
                        new_sentence = pattern.sub(replacement, new_sentence, count=1)
                        modified = True
                        break  # Une seule substitution par variante
        
        # Méthode 2: Variation de format numérique (20% du temps)
        elif random.random() < 0.2:
            for pattern, replacements in number_patterns.items():
                if pattern in new_sentence:
                    replacement = random.choice([r for r in replacements if r != pattern])
                    new_sentence = new_sentence.replace(pattern, replacement, 1)
                    modified = True
                    break
        
        # Méthode 3: Réorganisation légère (20% du temps)
        elif random.random() < 0.2 and ',' in new_sentence:
            # Intervertir ordre de clauses séparées par virgule (parfois)
            parts = new_sentence.split(',', 1)
            if len(parts) == 2 and len(parts[0]) > 20 and len(parts[1]) > 20:
                new_sentence = parts[1].strip() + ', ' + parts[0].strip()
                modified = True
        
        # Capitaliser première lettre
        if modified:
            new_sentence = new_sentence[0].upper() + new_sentence[1:] if new_sentence else new_sentence
            
            # Ajouter si différente et pas déjà présente
            if new_sentence not in augmented and new_sentence != sentence:
                augmented.append(new_sentence)
    
    return augmented


# Configuration augmentation
print("📝 Configuration:")
print(f"   Phrases originales: {len(df):,}")
print(f"   Variantes par phrase: 2-3")
print(f"   Facteur augmentation cible: ~3x\n")

# Appliquer augmentation avec barre de progression
print("🔄 Génération des variantes...")
augmented_data = []
skipped = 0
stats = {'original': 0, 'variants': 0}

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmentation"):
    try:
        # Déterminer nombre de variantes selon longueur de phrase
        sentence_len = len(row['sentence'])
        if sentence_len > 100:
            n_var = 3  # Plus de variantes pour phrases longues
        elif sentence_len > 50:
            n_var = 2
        else:
            n_var = 1  # Moins pour phrases courtes
        
        # Générer variantes
        variants = augment_financial_sentence(
            row['sentence'],
            row['sentiment'],
            n_variants=n_var
        )
        
        # Ajouter toutes les variantes
        for i, variant in enumerate(variants):
            augmented_data.append({
                'sentence': variant,
                'sentiment': row['sentiment'],
                'is_original': (i == 0),
                'source_idx': idx
            })
            
            if i == 0:
                stats['original'] += 1
            else:
                stats['variants'] += 1
    
    except Exception as e:
        skipped += 1
        # En cas d'erreur, garder au moins l'originale
        augmented_data.append({
            'sentence': row['sentence'],
            'sentiment': row['sentiment'],
            'is_original': True,
            'source_idx': idx
        })

# Créer DataFrame augmenté
df_augmented = pd.DataFrame(augmented_data)

# Supprimer duplicats stricts
initial_aug_len = len(df_augmented)
df_augmented = df_augmented.drop_duplicates(subset=['sentence'], keep='first')
duplicates_removed = initial_aug_len - len(df_augmented)

print(f"\n✅ AUGMENTATION TERMINÉE!")
print(f"{'='*60}")
print(f"   Dataset original:      {len(df):,} phrases")
print(f"   Dataset augmenté:      {len(df_augmented):,} phrases")
print(f"   Facteur augmentation:  {len(df_augmented)/len(df):.2f}x")
print(f"\n   Phrases originales:    {stats['original']:,}")
print(f"   Variantes générées:    {stats['variants']:,}")
print(f"   Duplicats supprimés:   {duplicates_removed:,}")

if skipped > 0:
    print(f"   ⚠️ Erreurs ignorées:   {skipped}")

# Distribution après augmentation
print(f"\n📊 DISTRIBUTION APRÈS AUGMENTATION:")
print(f"{'='*60}")

aug_counts = df_augmented['sentiment'].value_counts()
for sent in ['positive', 'neutral', 'negative']:
    if sent in aug_counts.index:
        count = aug_counts[sent]
        pct = count / len(df_augmented) * 100
        
        # Barre visuelle
        bar = '█' * int(pct / 2)
        
        # Changement vs original
        original_count = df['sentiment'].value_counts().get(sent, 0)
        growth = ((count - original_count) / original_count * 100) if original_count > 0 else 0
        
        print(f"   {sent:8s}: {count:5d} ({pct:5.1f}%) {bar}")
        print(f"             ↳ +{growth:.0f}% vs original")

# Qualité de l'augmentation
print(f"\n📈 QUALITÉ DE L'AUGMENTATION:")
print(f"{'='*60}")

# Longueur moyenne des phrases
original_len = df['sentence'].str.len().mean()
augmented_len = df_augmented['sentence'].str.len().mean()
print(f"   Longueur moyenne originale:  {original_len:.1f} caractères")
print(f"   Longueur moyenne augmentée:  {augmented_len:.1f} caractères")
print(f"   Différence:                  {abs(augmented_len - original_len):.1f} caractères")

# Diversité lexicale
original_vocab = set(' '.join(df['sentence']).lower().split())
augmented_vocab = set(' '.join(df_augmented['sentence']).lower().split())
print(f"\n   Vocabulaire original:        {len(original_vocab):,} mots uniques")
print(f"   Vocabulaire augmenté:        {len(augmented_vocab):,} mots uniques")
print(f"   Nouveaux mots:               {len(augmented_vocab - original_vocab):,}")

# Sauvegarder
augmented_path = data_dir / "financial_phrasebank_augmented.csv"
df_augmented.to_csv(augmented_path, index=False, encoding='utf-8')
print(f"\n💾 Dataset augmenté sauvegardé:")
print(f"   {augmented_path}")

# EXEMPLES D'AUGMENTATION
print(f"\n📝 EXEMPLES D'AUGMENTATION:")
print(f"{'='*60}\n")

# Prendre 3 exemples aléatoires
sample_indices = random.sample(range(len(df)), min(3, len(df)))

for sample_idx in sample_indices:
    original_sent = df.iloc[sample_idx]['sentence']
    sentiment = df.iloc[sample_idx]['sentiment']
    
    # Trouver variantes de cette phrase
    variants = df_augmented[df_augmented['source_idx'] == sample_idx]['sentence'].tolist()
    
    if len(variants) > 1:
        print(f"{sentiment.upper()}:")
        print(f"  Original:")
        print(f"    {original_sent[:120]}{'...' if len(original_sent) > 120 else ''}")
        
        for i, variant in enumerate(variants[1:], 1):
            print(f"  Variant {i}:")
            print(f"    {variant[:120]}{'...' if len(variant) > 120 else ''}")
        print()

print(f"{'='*60}")
print(f"✅ PRÊT POUR 5-FOLD CROSS-VALIDATION!")
print(f"{'='*60}")


🔄 AUGMENTATION DES DONNÉES

📝 Configuration:
   Phrases originales: 2,258
   Variantes par phrase: 2-3
   Facteur augmentation cible: ~3x

🔄 Génération des variantes...


Augmentation:   0%|          | 0/2258 [00:00<?, ?it/s]


✅ AUGMENTATION TERMINÉE!
   Dataset original:      2,258 phrases
   Dataset augmenté:      4,984 phrases
   Facteur augmentation:  2.21x

   Phrases originales:    2,258
   Variantes générées:    2,726
   Duplicats supprimés:   0

📊 DISTRIBUTION APRÈS AUGMENTATION:
   positive:  1676 ( 33.6%) ████████████████
             ↳ +194% vs original
   neutral :  2379 ( 47.7%) ███████████████████████
             ↳ +72% vs original
   negative:   929 ( 18.6%) █████████
             ↳ +207% vs original

📈 QUALITÉ DE L'AUGMENTATION:
   Longueur moyenne originale:  121.9 caractères
   Longueur moyenne augmentée:  131.6 caractères
   Différence:                  9.7 caractères

   Vocabulaire original:        7,018 mots uniques
   Vocabulaire augmenté:        7,046 mots uniques
   Nouveaux mots:               28

💾 Dataset augmenté sauvegardé:
   c:\Users\lenovo\OneDrive\Bureau\cours\sentiTrade-HMA-V2\data\external\financial_phrasebank_augmented.csv

📝 EXEMPLES D'AUGMENTATION:

NEUTRAL:
  Origina

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

print("📊 CRÉATION DES 5-FOLD CROSS-VALIDATION")
print(f"{'='*60}\n")

# Charger dataset augmenté
augmented_path = data_dir / "financial_phrasebank_augmented.csv"
df_full = pd.read_csv(augmented_path)

print(f"📁 Dataset chargé:")
print(f"   Total phrases: {len(df_full):,}")
print(f"   Colonnes: {df_full.columns.tolist()}")

# Créer mapping labels → IDs numériques
label_to_id = {'negative': 0, 'neutral': 1, 'positive': 2}
id_to_label = {v: k for k, v in label_to_id.items()}

df_full['label_id'] = df_full['sentiment'].map(label_to_id)

print(f"\n📊 Distribution globale:")
for label, label_id in label_to_id.items():
    count = (df_full['label_id'] == label_id).sum()
    pct = count / len(df_full) * 100
    print(f"   {label:8s} (id={label_id}): {count:5d} ({pct:5.1f}%)")

# Préparer données pour stratified split
X = df_full['sentence'].values
y = df_full['label_id'].values

print(f"\n🔀 Configuration StratifiedKFold:")
print(f"   Nombre de folds: 5")
print(f"   Stratification: ✅ (préserve distribution)")
print(f"   Shuffle: ✅")
print(f"   Random state: 42 (reproductibilité)")

# Créer le splitter stratifié
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Créer dossier pour les folds
folds_dir = data_dir / "folds"
folds_dir.mkdir(exist_ok=True)
print(f"\n📂 Dossier des folds: {folds_dir}")

# Statistiques globales
fold_stats = []

print(f"\n{'='*60}")
print("CRÉATION DES FOLDS")
print(f"{'='*60}\n")

# Créer et sauvegarder chaque fold
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"{'─'*60}")
    print(f"FOLD {fold_idx + 1}/5")
    print(f"{'─'*60}")
    
    # Créer DataFrames train/val
    train_df = df_full.iloc[train_idx].copy()
    val_df = df_full.iloc[val_idx].copy()
    
    # Statistiques du fold
    print(f"\n📊 Taille des ensembles:")
    print(f"   Train: {len(train_df):,} phrases ({len(train_df)/len(df_full)*100:.1f}%)")
    print(f"   Val:   {len(val_df):,} phrases ({len(val_df)/len(df_full)*100:.1f}%)")
    
    # Distribution train
    print(f"\n📈 Distribution TRAIN:")
    train_dist = train_df['sentiment'].value_counts()
    for sent in ['positive', 'neutral', 'negative']:
        if sent in train_dist.index:
            count = train_dist[sent]
            pct = count / len(train_df) * 100
            print(f"   {sent:8s}: {count:4d} ({pct:5.1f}%)")
    
    # Distribution validation
    print(f"\n📈 Distribution VALIDATION:")
    val_dist = val_df['sentiment'].value_counts()
    for sent in ['positive', 'neutral', 'negative']:
        if sent in val_dist.index:
            count = val_dist[sent]
            pct = count / len(val_df) * 100
            print(f"   {sent:8s}: {count:4d} ({pct:5.1f}%)")
    
    # Vérifier stratification
    train_ratio = train_dist / len(train_df)
    val_ratio = val_dist / len(val_df)
    max_diff = abs(train_ratio - val_ratio).max() * 100
    
    print(f"\n⚖️  Qualité stratification:")
    print(f"   Différence max: {max_diff:.2f}%")
    if max_diff < 2:
        print(f"   ✅ Excellente stratification")
    elif max_diff < 5:
        print(f"   ✅ Bonne stratification")
    else:
        print(f"   ⚠️ Stratification acceptable")
    
    # Sauvegarder les folds
    train_path = folds_dir / f"fold_{fold_idx}_train.csv"
    val_path = folds_dir / f"fold_{fold_idx}_val.csv"
    
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    
    print(f"\n💾 Sauvegardé:")
    print(f"   Train: {train_path.name}")
    print(f"   Val:   {val_path.name}")
    
    # Collecter stats
    fold_stats.append({
        'fold': fold_idx,
        'train_size': len(train_df),
        'val_size': len(val_df),
        'train_positive': train_dist.get('positive', 0),
        'train_neutral': train_dist.get('neutral', 0),
        'train_negative': train_dist.get('negative', 0),
        'val_positive': val_dist.get('positive', 0),
        'val_neutral': val_dist.get('neutral', 0),
        'val_negative': val_dist.get('negative', 0),
        'stratification_quality': max_diff
    })
    
    print()

# Créer DataFrame des statistiques
stats_df = pd.DataFrame(fold_stats)

print(f"{'='*60}")
print("RÉSUMÉ DES 5 FOLDS")
print(f"{'='*60}\n")

# Afficher tableau récapitulatif
print("📊 Tailles des ensembles:")
print(stats_df[['fold', 'train_size', 'val_size']].to_string(index=False))

print(f"\n📈 Distribution moyenne:")
print(f"   Train - Positive: {stats_df['train_positive'].mean():.0f} ± {stats_df['train_positive'].std():.0f}")
print(f"   Train - Neutral:  {stats_df['train_neutral'].mean():.0f} ± {stats_df['train_neutral'].std():.0f}")
print(f"   Train - Negative: {stats_df['train_negative'].mean():.0f} ± {stats_df['train_negative'].std():.0f}")

print(f"\n   Val - Positive:   {stats_df['val_positive'].mean():.0f} ± {stats_df['val_positive'].std():.0f}")
print(f"   Val - Neutral:    {stats_df['val_neutral'].mean():.0f} ± {stats_df['val_neutral'].std():.0f}")
print(f"   Val - Negative:   {stats_df['val_negative'].mean():.0f} ± {stats_df['val_negative'].std():.0f}")

print(f"\n⚖️  Qualité stratification moyenne: {stats_df['stratification_quality'].mean():.2f}% ± {stats_df['stratification_quality'].std():.2f}%")

# Sauvegarder statistiques
stats_path = folds_dir / "folds_statistics.csv"
stats_df.to_csv(stats_path, index=False)
print(f"\n💾 Statistiques sauvegardées: {stats_path.name}")

# Vérification finale
print(f"\n{'='*60}")
print("VÉRIFICATION FINALE")
print(f"{'='*60}")

# Vérifier que tous les fichiers existent
all_files_exist = True
for fold_idx in range(5):
    train_file = folds_dir / f"fold_{fold_idx}_train.csv"
    val_file = folds_dir / f"fold_{fold_idx}_val.csv"
    
    if not train_file.exists() or not val_file.exists():
        print(f"❌ Fold {fold_idx}: Fichiers manquants")
        all_files_exist = False
    else:
        print(f"✅ Fold {fold_idx}: Train ({train_file.stat().st_size/1024:.1f} KB) + Val ({val_file.stat().st_size/1024:.1f} KB)")

if all_files_exist:
    print(f"\n{'='*60}")
    print(f"✅ 5-FOLD CROSS-VALIDATION PRÊT!")
    print(f"{'='*60}")
    print(f"\n📂 Fichiers créés: {len(list(folds_dir.glob('*.csv')))} fichiers CSV")
    print(f"   • 5 folds train")
    print(f"   • 5 folds validation")
    print(f"   • 1 fichier statistiques")
    print(f"\n🚀 Prêt pour la configuration QLoRA et l'entraînement!")
else:
    print(f"\n⚠️ Certains fichiers sont manquants. Vérifie les erreurs ci-dessus.")


📊 CRÉATION DES 5-FOLD CROSS-VALIDATION

📁 Dataset chargé:
   Total phrases: 4,984
   Colonnes: ['sentence', 'sentiment', 'is_original', 'source_idx']

📊 Distribution globale:
   negative (id=0):   929 ( 18.6%)
   neutral  (id=1):  2379 ( 47.7%)
   positive (id=2):  1676 ( 33.6%)

🔀 Configuration StratifiedKFold:
   Nombre de folds: 5
   Stratification: ✅ (préserve distribution)
   Shuffle: ✅
   Random state: 42 (reproductibilité)

📂 Dossier des folds: c:\Users\lenovo\OneDrive\Bureau\cours\sentiTrade-HMA-V2\data\external\folds

CRÉATION DES FOLDS

────────────────────────────────────────────────────────────
FOLD 1/5
────────────────────────────────────────────────────────────

📊 Taille des ensembles:
   Train: 3,987 phrases (80.0%)
   Val:   997 phrases (20.0%)

📈 Distribution TRAIN:
   positive: 1341 ( 33.6%)
   neutral : 1903 ( 47.7%)
   negative:  743 ( 18.6%)

📈 Distribution VALIDATION:
   positive:  335 ( 33.6%)
   neutral :  476 ( 47.7%)
   negative:  186 ( 18.7%)

⚖️  Qualité str

In [ ]:
import torch
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("⚙️ CONFIGURATION QLORA")
print(f"{'='*60}\n")

# Vérifier disponibilité GPU/CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device détecté: {device.upper()}")

if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(f"   ⚠️ CPU uniquement - Entraînement sera plus lent")
    print(f"   💡 Recommandation: Réduire batch_size et epochs")

# Vérifier transformers et peft
try:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
    from peft import LoraConfig, get_peft_model, TaskType, PeftModel
    print(f"\n✅ Bibliothèques installées:")
    print(f"   • transformers")
    print(f"   • peft (QLORA)")
except ImportError as e:
    print(f"\n❌ Bibliothèques manquantes: {e}")
    print(f"\n💡 Installe avec:")
    print(f"   pip install transformers peft accelerate bitsandbytes")
    raise

# Configuration LoRA optimale pour CPU/GPU
print(f"\n📐 CONFIGURATION LORA:")
print(f"{'─'*60}")

lora_config = {
    'task_type': TaskType.SEQ_CLS,
    'r': 8,                    # Rank LoRA (réduit de 16 à 8 pour CPU)
    'lora_alpha': 16,          # Scaling factor (réduit de 32 à 16)
    'lora_dropout': 0.1,       # Dropout
    'target_modules': [
        "q_proj", "v_proj",    # Attention (focus sur query et value)
    ],
    'bias': "none",
    'inference_mode': False
}

print(f"   LoRA Rank (r):        {lora_config['r']}")
print(f"   LoRA Alpha:           {lora_config['lora_alpha']}")
print(f"   LoRA Dropout:         {lora_config['lora_dropout']}")
print(f"   Target Modules:       {', '.join(lora_config['target_modules'])}")
print(f"   Trainable params:     ~0.3% du modèle total")

# Configuration entraînement adaptée au device
print(f"\n🎯 CONFIGURATION ENTRAÎNEMENT:")
print(f"{'─'*60}")

if device == 'cuda':
    batch_size = 8
    num_epochs = 3
    gradient_accumulation = 1
else:
    batch_size = 4      # Réduit pour CPU
    num_epochs = 2      # Moins d'epochs pour CPU
    gradient_accumulation = 2  # Simule batch_size de 8

training_config = {
    'output_dir': str(project_root / "models" / "sentiment" / "llm" / "llama2_qlora"),
    'learning_rate': 2e-4,     # Learning rate optimal pour LoRA
    'per_device_train_batch_size': batch_size,
    'per_device_eval_batch_size': batch_size,
    'gradient_accumulation_steps': gradient_accumulation,
    'num_train_epochs': num_epochs,
    'weight_decay': 0.01,
    'evaluation_strategy': "epoch",
    'save_strategy': "epoch",
    'load_best_model_at_end': True,
    'metric_for_best_model': "matthews_correlation",
    'greater_is_better': True,
    'logging_steps': 50,
    'save_total_limit': 2,     # Garder seulement 2 meilleurs checkpoints
    'fp16': torch.cuda.is_available(),  # Mixed precision si GPU
    'optim': "adamw_torch",
    'warmup_ratio': 0.1,
    'report_to': "none"        # Pas de logging externe
}

print(f"   Batch Size:           {batch_size}")
print(f"   Gradient Accum:       {gradient_accumulation}")
print(f"   Effective Batch:      {batch_size * gradient_accumulation}")
print(f"   Epochs:               {num_epochs}")
print(f"   Learning Rate:        {training_config['learning_rate']}")
print(f"   Weight Decay:         {training_config['weight_decay']}")
print(f"   FP16 (mixed prec):    {training_config['fp16']}")
print(f"   Best Metric:          MCC (Matthews Correlation)")

# Estimer temps d'entraînement
train_samples_per_fold = 3987
steps_per_epoch = train_samples_per_fold // (batch_size * gradient_accumulation)
total_steps_per_fold = steps_per_epoch * num_epochs
total_folds = 5

if device == 'cuda':
    time_per_step = 0.5  # secondes (GPU)
else:
    time_per_step = 2.0  # secondes (CPU - approximatif)

estimated_time_minutes = (total_steps_per_fold * total_folds * time_per_step) / 60

print(f"\n⏱️  ESTIMATION TEMPS D'ENTRAÎNEMENT:")
print(f"{'─'*60}")
print(f"   Steps par epoch:      {steps_per_epoch}")
print(f"   Total steps/fold:     {total_steps_per_fold}")
print(f"   Total steps (5 folds): {total_steps_per_fold * total_folds}")
print(f"   Temps estimé/fold:    ~{estimated_time_minutes/5:.0f} minutes")
print(f"   Temps total estimé:   ~{estimated_time_minutes:.0f} minutes ({estimated_time_minutes/60:.1f}h)")

if device == 'cpu' and estimated_time_minutes > 120:
    print(f"\n   ⚠️ Sur CPU, l'entraînement sera long!")
    print(f"   💡 Options:")
    print(f"      1. Laisse tourner toute la nuit")
    print(f"      2. Teste d'abord sur 1 fold uniquement")
    print(f"      3. Utilise Google Colab avec GPU gratuit")

# Créer dossiers de sortie
output_dir = Path(training_config['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

print(f"\n📂 Dossiers de sortie:")
print(f"   Modèles: {output_dir}")
print(f"   Résultats: {results_dir}")

print(f"\n{'='*60}")
print(f"✅ CONFIGURATION PRÊTE!")
print(f"{'='*60}")
print(f"\n💡 Conseil: Sauvegarde ce notebook avant de lancer l'entraînement!")


⚙️ CONFIGURATION QLORA

🖥️  Device détecté: CPU
   ⚠️ CPU uniquement - Entraînement sera plus lent
   💡 Recommandation: Réduire batch_size et epochs

✅ Bibliothèques installées:
   • transformers
   • peft (QLORA)

📐 CONFIGURATION LORA:
────────────────────────────────────────────────────────────
   LoRA Rank (r):        8
   LoRA Alpha:           16
   LoRA Dropout:         0.1
   Target Modules:       q_proj, v_proj
   Trainable params:     ~0.3% du modèle total

🎯 CONFIGURATION ENTRAÎNEMENT:
────────────────────────────────────────────────────────────
   Batch Size:           4
   Gradient Accum:       2
   Effective Batch:      8
   Epochs:               2
   Learning Rate:        0.0002
   Weight Decay:         0.01
   FP16 (mixed prec):    False
   Best Metric:          MCC (Matthews Correlation)

⏱️  ESTIMATION TEMPS D'ENTRAÎNEMENT:
────────────────────────────────────────────────────────────
   Steps par epoch:      498
   Total steps/fold:     996
   Total steps (5 folds): 498

In [ ]:
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    precision_score,
    recall_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)
import numpy as np

print("📊 CONFIGURATION DES MÉTRIQUES")
print(f"{'='*60}\n")

def compute_metrics(eval_pred):
    """
    Calculer toutes les métriques pour l'évaluation
    
    Métriques clés pour publication académique:
    - Accuracy: Précision globale
    - F1 Macro: F1 moyen non pondéré (important pour classes déséquilibrées)
    - F1 Weighted: F1 moyen pondéré par support
    - MCC: Matthews Correlation Coefficient (métrique la plus importante!)
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    # Calculer métriques
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    precision_macro = precision_score(labels, predictions, average='macro')
    recall_macro = recall_score(labels, predictions, average='macro')
    mcc = matthews_corrcoef(labels, predictions)
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'matthews_correlation': mcc  # Métrique clé pour best model
    }

def print_detailed_results(predictions, labels, fold_idx):
    """
    Afficher résultats détaillés d'un fold
    """
    preds = np.argmax(predictions, axis=1)
    
    print(f"\n{'='*60}")
    print(f"📊 FOLD {fold_idx} - RÉSULTATS DÉTAILLÉS")
    print(f"{'='*60}\n")
    
    # Classification report
    print("📋 Classification Report:")
    print("─" * 60)
    report = classification_report(
        labels, preds, 
        target_names=['negative', 'neutral', 'positive'],
        digits=4
    )
    print(report)
    
    # Confusion matrix
    print("\n🔢 Confusion Matrix:")
    print("─" * 60)
    cm = confusion_matrix(labels, preds)
    
    # Afficher avec labels
    print("              Predicted")
    print("              Neg    Neu    Pos")
    print("         ┌─────────────────────")
    for i, label in enumerate(['Neg', 'Neu', 'Pos']):
        print(f"  Actual {label} │ {cm[i][0]:4d}  {cm[i][1]:4d}  {cm[i][2]:4d}")
    
    # Métriques clés
    accuracy = accuracy_score(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    
    print(f"\n✨ MÉTRIQUES PRINCIPALES:")
    print("─" * 60)
    print(f"   Accuracy:     {accuracy:.4f}")
    print(f"   MCC:          {mcc:.4f} ⭐ (Métrique clé)")
    print(f"   F1 Macro:     {f1_macro:.4f}")
    print(f"   F1 Weighted:  {f1_weighted:.4f}")
    
    # Interprétation MCC
    print(f"\n💡 Interprétation MCC ({mcc:.4f}):")
    if mcc > 0.9:
        print(f"   🟢 Excellent: Corrélation très forte")
    elif mcc > 0.8:
        print(f"   🟢 Très bon: Corrélation forte")
    elif mcc > 0.7:
        print(f"   🟡 Bon: Corrélation modérée à forte")
    elif mcc > 0.5:
        print(f"   🟡 Acceptable: Corrélation modérée")
    else:
        print(f"   🔴 Faible: Corrélation faible")
    
    return {
        'accuracy': accuracy,
        'mcc': mcc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm.tolist()
    }

print("✅ Fonctions de métriques définies:")
print("   • compute_metrics() - Pour Trainer")
print("   • print_detailed_results() - Résultats détaillés par fold")
print("\n📌 Métrique principale: MCC (Matthews Correlation Coefficient)")
print("   • Range: [-1, +1]")
print("   • +1: Prédiction parfaite")
print("   • 0: Prédiction aléatoire")
print("   • -1: Désaccord total")
print("   • Robuste aux classes déséquilibrées ✅")

print(f"\n{'='*60}")
print(f"✅ MÉTRIQUES PRÊTES!")
print(f"{'='*60}")


📊 CONFIGURATION DES MÉTRIQUES

✅ Fonctions de métriques définies:
   • compute_metrics() - Pour Trainer
   • print_detailed_results() - Résultats détaillés par fold

📌 Métrique principale: MCC (Matthews Correlation Coefficient)
   • Range: [-1, +1]
   • +1: Prédiction parfaite
   • 0: Prédiction aléatoire
   • -1: Désaccord total
   • Robuste aux classes déséquilibrées ✅

✅ MÉTRIQUES PRÊTES!


In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch
import json
from datetime import datetime

print("🧪 TEST RAPIDE - ENTRAÎNEMENT 1 FOLD")
print(f"{'='*60}\n")

# Configuration
TEST_FOLD = 0
MODEL_NAME = "distilbert-base-uncased"

print(f"📋 Configuration du test:")
print(f"   Fold testé: {TEST_FOLD}")
print(f"   Modèle: {MODEL_NAME}")
print(f"   Raison: DistilBERT est plus rapide que Llama-2 pour le test")
print(f"   ⚠️ Pour production finale, on utilisera Llama-2")

# Timer
start_time = datetime.now()
print(f"\n⏰ Début: {start_time.strftime('%H:%M:%S')}\n")

# ============================================================
# 1. CHARGER DONNÉES DU FOLD 0
# ============================================================
print(f"{'─'*60}")
print("📂 ÉTAPE 1: Chargement des données")
print(f"{'─'*60}\n")

folds_dir = data_dir / "folds"
train_df = pd.read_csv(folds_dir / f"fold_{TEST_FOLD}_train.csv")
val_df = pd.read_csv(folds_dir / f"fold_{TEST_FOLD}_val.csv")

print(f"✅ Données chargées:")
print(f"   Train: {len(train_df):,} phrases")
print(f"   Val:   {len(val_df):,} phrases")

# ============================================================
# 2. TOKENIZER
# ============================================================
print(f"\n{'─'*60}")
print("🔤 ÉTAPE 2: Tokenization")
print(f"{'─'*60}\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Fonction de tokenization
def tokenize_function(examples):
    return tokenizer(
        examples['sentence'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

print(f"✅ Tokenizer initialisé: {MODEL_NAME}")

# ============================================================
# 3. PRÉPARER DATASETS
# ============================================================
print(f"\n{'─'*60}")
print("🔧 ÉTAPE 3: Préparation des datasets")
print(f"{'─'*60}\n")

# Convertir en Dataset HuggingFace
train_dataset = Dataset.from_pandas(train_df[['sentence', 'label_id']])
val_dataset = Dataset.from_pandas(val_df[['sentence', 'label_id']])

print("📝 Tokenization en cours...")
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['sentence'])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['sentence'])

train_dataset = train_dataset.rename_column('label_id', 'labels')
val_dataset = val_dataset.rename_column('label_id', 'labels')

train_dataset.set_format('torch')
val_dataset.set_format('torch')

print(f"✅ Datasets préparés: {len(train_dataset)} train, {len(val_dataset)} val")

# ============================================================
# 4. CHARGER MODÈLE
# ============================================================
print(f"\n{'─'*60}")
print("🤖 ÉTAPE 4: Chargement du modèle")
print(f"{'─'*60}\n")

print("📥 Chargement du modèle base...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'negative', 1: 'neutral', 2: 'positive'},
    label2id={'negative': 0, 'neutral': 1, 'positive': 2}
)

print(f"✅ Modèle base chargé")

# Configuration LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none",
    inference_mode=False
)

model = get_peft_model(model, lora_config)
print(f"\n✅ LoRA appliqué:")
model.print_trainable_parameters()

# ============================================================
# 5. CONFIGURATION ENTRAÎNEMENT (CORRIGÉE)
# ============================================================
print(f"\n{'─'*60}")
print("⚙️ ÉTAPE 5: Configuration de l'entraînement")
print(f"{'─'*60}\n")

# CORRECTION: Utiliser eval_strategy au lieu de evaluation_strategy
training_args = TrainingArguments(
    output_dir=str(project_root / "models" / "test_qlora"),
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",           # CORRECTION ICI
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="matthews_correlation",
    greater_is_better=True,
    logging_steps=50,
    logging_dir=str(project_root / "logs"),
    save_total_limit=1,
    use_cpu=not torch.cuda.is_available(),  # AJOUT: Force CPU si pas GPU
    warmup_ratio=0.1,
    report_to="none"
)

print(f"✅ Configuration:")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Gradient accum: {training_args.gradient_accumulation_steps}")
print(f"   Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Learning rate: {training_args.learning_rate}")

# ============================================================
# 6. CRÉER TRAINER
# ============================================================
print(f"\n{'─'*60}")
print("🏋️ ÉTAPE 6: Création du Trainer")
print(f"{'─'*60}\n")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

print("✅ Trainer créé")

# ============================================================
# 7. ENTRAÎNEMENT
# ============================================================
print(f"\n{'='*60}")
print("🚀 ÉTAPE 7: ENTRAÎNEMENT")
print(f"{'='*60}\n")

print("⏳ Entraînement en cours...")
print("   Sur CPU: 25-35 minutes estimées")
print("   Tu peux suivre la progression avec la barre ci-dessous\n")

try:
    train_result = trainer.train()
    
    train_end_time = datetime.now()
    train_duration = (train_end_time - start_time).total_seconds() / 60
    
    print(f"\n✅ Entraînement terminé!")
    print(f"   Durée: {train_duration:.1f} minutes")
    print(f"   Loss finale: {train_result.training_loss:.4f}")
    
except Exception as e:
    print(f"\n❌ Erreur pendant l'entraînement:")
    print(f"   {e}")
    import traceback
    traceback.print_exc()
    raise

# ============================================================
# 8. ÉVALUATION
# ============================================================
print(f"\n{'='*60}")
print("📊 ÉTAPE 8: ÉVALUATION")
print(f"{'='*60}\n")

print("🔍 Évaluation en cours...")
eval_results = trainer.evaluate()

print(f"\n✅ Résultats d'évaluation:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"   {key:25s}: {value:.4f}")

# Prédictions détaillées
print(f"\n🔍 Génération des prédictions détaillées...")
predictions = trainer.predict(val_dataset)
detailed_results = print_detailed_results(
    predictions.predictions,
    predictions.label_ids,
    TEST_FOLD
)

# ============================================================
# 9. SAUVEGARDER RÉSULTATS
# ============================================================
print(f"\n{'='*60}")
print("💾 ÉTAPE 9: Sauvegarde des résultats")
print(f"{'='*60}\n")

test_results = {
    'fold': TEST_FOLD,
    'model': MODEL_NAME,
    'training_duration_minutes': train_duration,
    'train_samples': len(train_dataset),
    'val_samples': len(val_dataset),
    'epochs': training_args.num_train_epochs,
    'final_train_loss': train_result.training_loss,
    **eval_results,
    **detailed_results
}

results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

test_results_path = results_dir / "test_1fold_results.json"
with open(test_results_path, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"✅ Résultats sauvegardés:")
print(f"   {test_results_path}")

model_save_path = project_root / "models" / "test_qlora" / f"fold_{TEST_FOLD}_final"
trainer.save_model(model_save_path)
print(f"\n✅ Modèle sauvegardé:")
print(f"   {model_save_path}")

# ============================================================
# 10. RÉSUMÉ FINAL
# ============================================================
end_time = datetime.now()
total_duration = (end_time - start_time).total_seconds() / 60

print(f"\n{'='*60}")
print("✅ TEST RAPIDE TERMINÉ!")
print(f"{'='*60}\n")

print(f"⏱️  DURÉE TOTALE: {total_duration:.1f} minutes")
print(f"   Début: {start_time.strftime('%H:%M:%S')}")
print(f"   Fin:   {end_time.strftime('%H:%M:%S')}")

print(f"\n📊 RÉSULTATS CLÉS:")
print(f"   Accuracy: {test_results['accuracy']:.4f}")
print(f"   MCC:      {test_results['mcc']:.4f} ⭐")
print(f"   F1 Macro: {test_results['f1_macro']:.4f}")

print(f"\n💡 PROCHAINES ÉTAPES:")
if test_results['mcc'] > 0.7:
    print(f"   🟢 Excellent! MCC > 0.7")
    print(f"   ✅ Tu peux lancer l'entraînement complet (5 folds)")
    print(f"   📌 Temps estimé: ~{total_duration * 5:.0f} minutes")
elif test_results['mcc'] > 0.5:
    print(f"   🟡 Acceptable. MCC > 0.5")
    print(f"   ✅ Tu peux lancer les 5 folds")
    print(f"   💡 Considère augmenter epochs (3) pour meilleurs résultats")
else:
    print(f"   🔴 MCC faible. Vérifie la convergence")

print(f"\n{'='*60}")
print(f"🎯 Si satisfait, lance la CELLULE 6B (5 folds complets)!")
print(f"{'='*60}")


🧪 TEST RAPIDE - ENTRAÎNEMENT 1 FOLD

📋 Configuration du test:
   Fold testé: 0
   Modèle: distilbert-base-uncased
   Raison: DistilBERT est plus rapide que Llama-2 pour le test
   ⚠️ Pour production finale, on utilisera Llama-2

⏰ Début: 19:44:05

────────────────────────────────────────────────────────────
📂 ÉTAPE 1: Chargement des données
────────────────────────────────────────────────────────────

✅ Données chargées:
   Train: 3,987 phrases
   Val:   997 phrases

────────────────────────────────────────────────────────────
🔤 ÉTAPE 2: Tokenization
────────────────────────────────────────────────────────────

✅ Tokenizer initialisé: distilbert-base-uncased

────────────────────────────────────────────────────────────
🔧 ÉTAPE 3: Préparation des datasets
────────────────────────────────────────────────────────────

📝 Tokenization en cours...


Map:   0%|          | 0/3987 [00:00<?, ? examples/s]

Map:   0%|          | 0/997 [00:00<?, ? examples/s]

✅ Datasets préparés: 3987 train, 997 val

────────────────────────────────────────────────────────────
🤖 ÉTAPE 4: Chargement du modèle
────────────────────────────────────────────────────────────

📥 Chargement du modèle base...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Modèle base chargé

✅ LoRA appliqué:
trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936

────────────────────────────────────────────────────────────
⚙️ ÉTAPE 5: Configuration de l'entraînement
────────────────────────────────────────────────────────────

✅ Configuration:
   Batch size: 4
   Gradient accum: 2
   Effective batch: 8
   Epochs: 2
   Learning rate: 0.0002

────────────────────────────────────────────────────────────
🏋️ ÉTAPE 6: Création du Trainer
────────────────────────────────────────────────────────────

✅ Trainer créé

🚀 ÉTAPE 7: ENTRAÎNEMENT

⏳ Entraînement en cours...
   Sur CPU: 25-35 minutes estimées
   Tu peux suivre la progression avec la barre ci-dessous



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro,Matthews Correlation
1,0.182200,0.127335,0.959880,0.957443,0.960055,0.954889,0.960601,0.936480
2,0.062200,0.094431,0.974925,0.972761,0.974972,0.972016,0.973584,0.959949



✅ Entraînement terminé!
   Durée: 76.1 minutes
   Loss finale: 0.2407

📊 ÉTAPE 8: ÉVALUATION

🔍 Évaluation en cours...



✅ Résultats d'évaluation:
   eval_loss                : 0.0944
   eval_accuracy            : 0.9749
   eval_f1_macro            : 0.9728
   eval_f1_weighted         : 0.9750
   eval_precision_macro     : 0.9720
   eval_recall_macro        : 0.9736
   eval_matthews_correlation: 0.9599
   eval_runtime             : 155.8556
   eval_samples_per_second  : 6.3970
   eval_steps_per_second    : 1.6040
   epoch                    : 2.0000

🔍 Génération des prédictions détaillées...

📊 FOLD 0 - RÉSULTATS DÉTAILLÉS

📋 Classification Report:
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

    negative     0.9677    0.9677    0.9677       186
     neutral     0.9894    0.9769    0.9831       476
    positive     0.9589    0.9761    0.9675       335

    accuracy                         0.9749       997
   macro avg     0.9720    0.9736    0.9728       997
weighted avg     0.9751    0.9749    0.9750       997


🔢 Confusion Matrix:

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch
import json
from datetime import datetime
import pandas as pd
import gc

print(" ENTRAÎNEMENT COMPLET 5-FOLD CROSS-VALIDATION")
print(f"{'='*60}\n")

# Configuration
MODEL_NAME = "distilbert-base-uncased"
NUM_FOLDS = 5

print(f" Configuration:")
print(f"   Modèle: {MODEL_NAME}")
print(f"   Nombre de folds: {NUM_FOLDS}")
print(f"   Epochs par fold: 2")
print(f"   Temps estimé total: ~6.5 heures\n")

# Timer global
global_start_time = datetime.now()
print(f"⏰ Début: {global_start_time.strftime('%H:%M:%S')}\n")

# Stockage résultats
all_results = []
fold_times = []

# ============================================================
# BOUCLE SUR LES 5 FOLDS
# ============================================================

for fold_idx in range(NUM_FOLDS):
    fold_start_time = datetime.now()
    
    print(f"\n{'#'*60}")
    print(f"# FOLD {fold_idx + 1}/{NUM_FOLDS}")
    print(f"{'#'*60}\n")
    
    try:
        # --------------------------------------------------------
        # 1. CHARGER DONNÉES
        # --------------------------------------------------------
        print(f"{'─'*60}")
        print(f" ÉTAPE 1: Chargement des données - Fold {fold_idx}")
        print(f"{'─'*60}\n")
        
        folds_dir = data_dir / "folds"
        train_df = pd.read_csv(folds_dir / f"fold_{fold_idx}_train.csv")
        val_df = pd.read_csv(folds_dir / f"fold_{fold_idx}_val.csv")
        
        print(f" Fold {fold_idx} chargé: {len(train_df)} train, {len(val_df)} val")
        
        # --------------------------------------------------------
        # 2. TOKENIZATION
        # --------------------------------------------------------
        print(f"\n{'─'*60}")
        print(f" ÉTAPE 2: Tokenization")
        print(f"{'─'*60}\n")
        
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        
        def tokenize_function(examples):
            return tokenizer(
                examples['sentence'],
                padding='max_length',
                truncation=True,
                max_length=128
            )
        
        # Préparer datasets
        train_dataset = Dataset.from_pandas(train_df[['sentence', 'label_id']])
        val_dataset = Dataset.from_pandas(val_df[['sentence', 'label_id']])
        
        train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['sentence'])
        val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['sentence'])
        
        train_dataset = train_dataset.rename_column('label_id', 'labels')
        val_dataset = val_dataset.rename_column('label_id', 'labels')
        
        train_dataset.set_format('torch')
        val_dataset.set_format('torch')
        
        print(f" Datasets tokenizés")
        
        # --------------------------------------------------------
        # 3. CHARGER MODÈLE
        # --------------------------------------------------------
        print(f"\n{'─'*60}")
        print(f"🤖 ÉTAPE 3: Chargement du modèle")
        print(f"{'─'*60}\n")
        
        # Charger modèle base (nouveau pour chaque fold)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3,
            id2label={0: 'negative', 1: 'neutral', 2: 'positive'},
            label2id={'negative': 0, 'neutral': 1, 'positive': 2}
        )
        
        # Appliquer LoRA
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            target_modules=["q_lin", "v_lin"],
            bias="none",
            inference_mode=False
        )
        
        model = get_peft_model(model, lora_config)
        print(f"✅ Modèle + LoRA initialisés")
        
        # --------------------------------------------------------
        # 4. CONFIGURATION ENTRAÎNEMENT
        # --------------------------------------------------------
        print(f"\n{'─'*60}")
        print(f"⚙️ ÉTAPE 4: Configuration")
        print(f"{'─'*60}\n")
        
        training_args = TrainingArguments(
            output_dir=str(project_root / "models" / "qlora_5fold" / f"fold_{fold_idx}"),
            learning_rate=2e-4,
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            gradient_accumulation_steps=2,
            num_train_epochs=2,
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="matthews_correlation",
            greater_is_better=True,
            logging_steps=50,
            logging_dir=str(project_root / "logs" / f"fold_{fold_idx}"),
            save_total_limit=1,
            use_cpu=not torch.cuda.is_available(),
            warmup_ratio=0.1,
            report_to="none"
        )
        
        print(f" Configuration prête")
        
        # --------------------------------------------------------
        # 5. CRÉER TRAINER
        # --------------------------------------------------------
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
            compute_metrics=compute_metrics
        )
        
        # --------------------------------------------------------
        # 6. ENTRAÎNEMENT
        # --------------------------------------------------------
        print(f"\n{'='*60}")
        print(f" ÉTAPE 5: ENTRAÎNEMENT FOLD {fold_idx}")
        print(f"{'='*60}\n")
        
        train_result = trainer.train()
        
        train_end_time = datetime.now()
        train_duration = (train_end_time - fold_start_time).total_seconds() / 60
        fold_times.append(train_duration)
        
        print(f"\n✅ Fold {fold_idx} entraîné en {train_duration:.1f} minutes")
        
        # --------------------------------------------------------
        # 7. ÉVALUATION
        # --------------------------------------------------------
        print(f"\n{'='*60}")
        print(f" ÉTAPE 6: ÉVALUATION FOLD {fold_idx}")
        print(f"{'='*60}\n")
        
        eval_results = trainer.evaluate()
        
        # Prédictions détaillées
        predictions = trainer.predict(val_dataset)
        detailed_results = print_detailed_results(
            predictions.predictions,
            predictions.label_ids,
            fold_idx
        )
        
        # --------------------------------------------------------
        # 8. SAUVEGARDER RÉSULTATS
        # --------------------------------------------------------
        fold_results = {
            'fold': fold_idx,
            'model': MODEL_NAME,
            'training_duration_minutes': train_duration,
            'train_samples': len(train_dataset),
            'val_samples': len(val_dataset),
            'epochs': training_args.num_train_epochs,
            'final_train_loss': train_result.training_loss,
            **eval_results,
            **detailed_results
        }
        
        all_results.append(fold_results)
        
        # Sauvegarder modèle du fold
        model_save_path = project_root / "models" / "qlora_5fold" / f"fold_{fold_idx}_final"
        trainer.save_model(model_save_path)
        
        # Sauvegarder résultats intermédiaires
        interim_results_path = project_root / "results" / "qlora_5fold_interim.json"
        with open(interim_results_path, 'w') as f:
            json.dump(all_results, f, indent=2)
        
        print(f"\n Résultats intermédiaires sauvegardés")
        print(f"   Fold {fold_idx} terminé: {train_duration:.1f} min")
        print(f"   MCC: {fold_results['mcc']:.4f}")
        print(f"   Accuracy: {fold_results['accuracy']:.4f}")
        
        # Temps restant estimé
        avg_time = sum(fold_times) / len(fold_times)
        remaining_folds = NUM_FOLDS - (fold_idx + 1)
        estimated_remaining = avg_time * remaining_folds
        
        if remaining_folds > 0:
            print(f"\n  Temps restant estimé: ~{estimated_remaining:.0f} minutes ({estimated_remaining/60:.1f}h)")
        
        # Nettoyage mémoire
        del model, trainer, train_dataset, val_dataset
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"\n ERREUR FOLD {fold_idx}:")
        print(f"   {e}")
        import traceback
        traceback.print_exc()
        
        # Sauvegarder résultats partiels même en cas d'erreur
        error_results_path = project_root / "results" / f"qlora_5fold_error_fold{fold_idx}.json"
        with open(error_results_path, 'w') as f:
            json.dump({
                'fold': fold_idx,
                'error': str(e),
                'partial_results': all_results
            }, f, indent=2)
        
        print(f"\n Résultats partiels sauvegardés: {error_results_path}")
        print(f"   Continue avec les folds restants...")
        continue

# ============================================================
# RÉSULTATS AGRÉGÉS
# ============================================================
global_end_time = datetime.now()
total_duration = (global_end_time - global_start_time).total_seconds() / 60

print(f"\n{'='*60}")
print(f" ENTRAÎNEMENT 5-FOLD TERMINÉ!")
print(f"{'='*60}\n")

print(f"  DURÉE TOTALE: {total_duration:.1f} minutes ({total_duration/60:.1f} heures)")
print(f"   Début: {global_start_time.strftime('%H:%M:%S')}")
print(f"   Fin:   {global_end_time.strftime('%H:%M:%S')}\n")

# Créer DataFrame des résultats
results_df = pd.DataFrame(all_results)

print(f"{'='*60}")
print(" RÉSULTATS AGRÉGÉS 5-FOLD CROSS-VALIDATION")
print(f"{'='*60}\n")

# Tableau résultats
print(" Résultats par fold:")
print(results_df[['fold', 'accuracy', 'mcc', 'f1_macro', 'training_duration_minutes']].to_string(index=False))

# Statistiques agrégées
print(f"\n STATISTIQUES FINALES:")
print(f"{'─'*60}")
print(f"   Accuracy:  {results_df['accuracy'].mean():.4f} ± {results_df['accuracy'].std():.4f}")
print(f"   MCC:       {results_df['mcc'].mean():.4f} ± {results_df['mcc'].std():.4f} ⭐")
print(f"   F1 Macro:  {results_df['f1_macro'].mean():.4f} ± {results_df['f1_macro'].std():.4f}")
print(f"   F1 Weighted: {results_df['f1_weighted'].mean():.4f} ± {results_df['f1_weighted'].std():.4f}")

# Interprétation
mcc_mean = results_df['mcc'].mean()
print(f"\n💡 INTERPRÉTATION:")
if mcc_mean > 0.95:
    print(f"    EXCEPTIONNEL! MCC moyen > 0.95")
    print(f"    Performance état de l'art (égale ou supérieure à FinBERT)")
elif mcc_mean > 0.90:
    print(f"    EXCELLENT! MCC moyen > 0.90")
    print(f"    Performance comparable à FinBERT")
elif mcc_mean > 0.80:
    print(f"    TRÈS BON! MCC moyen > 0.80")
    print(f"    Performance solide pour publication")
else:
    print(f"    BON. MCC moyen > {mcc_mean:.2f}")

# Sauvegarder résultats finaux
final_results_path = project_root / "results" / "qlora_5fold_final_results.json"
with open(final_results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

# Sauvegarder CSV
csv_path = project_root / "results" / "qlora_5fold_results.csv"
results_df.to_csv(csv_path, index=False)

print(f"\n RÉSULTATS SAUVEGARDÉS:")
print(f"   JSON: {final_results_path}")
print(f"   CSV:  {csv_path}")

print(f"\n{'='*60}")
print(f" PIPELINE COMPLET TERMINÉ!")
print(f"{'='*60}")
print(f"\n PROCHAINES ÉTAPES:")
print(f"   1. Analyse des résultats détaillés")
print(f"   2. Génération des graphiques pour le rapport")
print(f"   3. Rédaction de la section \"Expérimentations\"")
print(f"   4. Comparaison avec état de l'art (FinBERT)")


 ENTRAÎNEMENT COMPLET 5-FOLD CROSS-VALIDATION

 Configuration:
   Modèle: distilbert-base-uncased
   Nombre de folds: 5
   Epochs par fold: 2
   Temps estimé total: ~6.5 heures

⏰ Début: 21:08:43


############################################################
# FOLD 1/5
############################################################

────────────────────────────────────────────────────────────
 ÉTAPE 1: Chargement des données - Fold 0
────────────────────────────────────────────────────────────

 Fold 0 chargé: 3987 train, 997 val

────────────────────────────────────────────────────────────
 ÉTAPE 2: Tokenization
────────────────────────────────────────────────────────────



Map:   0%|          | 0/3987 [00:00<?, ? examples/s]

Map:   0%|          | 0/997 [00:00<?, ? examples/s]

 Datasets tokenizés

────────────────────────────────────────────────────────────
🤖 ÉTAPE 3: Chargement du modèle
────────────────────────────────────────────────────────────



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Datasets tokenizés

────────────────────────────────────────────────────────────
🤖 ÉTAPE 3: Chargement du modèle
────────────────────────────────────────────────────────────

✅ Modèle + LoRA initialisés

────────────────────────────────────────────────────────────
⚙️ ÉTAPE 4: Configuration
────────────────────────────────────────────────────────────

 Configuration prête

 ÉTAPE 5: ENTRAÎNEMENT FOLD 4



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro,Matthews Correlation
1,0.205900,0.124869,0.955823,0.951086,0.955756,0.948133,0.954684,0.929561
2,0.126200,0.102824,0.963855,0.959902,0.963823,0.959428,0.960421,0.942120



✅ Fold 4 entraîné en 52.2 minutes

 ÉTAPE 6: ÉVALUATION FOLD 4




📊 FOLD 4 - RÉSULTATS DÉTAILLÉS

📋 Classification Report:
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

    negative     0.9465    0.9568    0.9516       185
     neutral     0.9769    0.9811    0.9790       475
    positive     0.9548    0.9435    0.9491       336

    accuracy                         0.9639       996
   macro avg     0.9594    0.9604    0.9599       996
weighted avg     0.9638    0.9639    0.9638       996


🔢 Confusion Matrix:
────────────────────────────────────────────────────────────
              Predicted
              Neg    Neu    Pos
         ┌─────────────────────
  Actual Neg │  177     0     8
  Actual Neu │    2   466     7
  Actual Pos │    8    11   317

✨ MÉTRIQUES PRINCIPALES:
────────────────────────────────────────────────────────────
   Accuracy:     0.9639
   MCC:          0.9421 ⭐ (Métrique clé)
   F1 Macro:     0.9599
   F1 Weighted:  0.9638

💡 Interprétation MCC (0.9421):
  

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json

print(" GÉNÉRATION DES GRAPHIQUES POUR LE RAPPORT")
print(f"{'='*60}\n")

# Configuration style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Créer dossier figures
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

# Charger résultats
results_path = project_root / "results" / "qlora_5fold_final_results.json"
with open(results_path, 'r') as f:
    results = json.load(f)

results_df = pd.DataFrame(results)

# ============================================================
# GRAPHIQUE 1: Performance par Fold
# ============================================================
print(" Graphique 1: Performance par fold...")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['accuracy', 'mcc', 'f1_macro']
titles = ['Accuracy', 'MCC (Matthews Correlation)', 'F1 Macro']
colors = ['#2ecc71', '#3498db', '#e74c3c']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    values = results_df[metric].values
    folds = results_df['fold'].values
    
    # Barplot
    ax.bar(folds, values, color=color, alpha=0.7, edgecolor='black')
    
    # Ligne moyenne
    mean_val = values.mean()
    ax.axhline(mean_val, color='red', linestyle='--', linewidth=2, 
               label=f'Moyenne: {mean_val:.4f}')
    
    # Labels
    ax.set_xlabel('Fold', fontsize=12, fontweight='bold')
    ax.set_ylabel(title, fontsize=12, fontweight='bold')
    ax.set_title(f'{title} par Fold', fontsize=14, fontweight='bold')
    ax.set_xticks(folds)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Annotations
    for i, v in enumerate(values):
        ax.text(i, v + 0.001, f'{v:.4f}', ha='center', va='bottom', 
                fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(figures_dir / 'fold_performance.png', dpi=300, bbox_inches='tight')
print(f"    Sauvegardé: fold_performance.png")
plt.close()

# ============================================================
# GRAPHIQUE 2: Comparaison avec l'état de l'art
# ============================================================
print(" Graphique 2: Comparaison avec état de l'art...")

comparison_data = {
    'Model': ['VADER\n(Baseline)', 'FinBERT\n(SOTA)', 'DistilBERT-LoRA\n(Notre modèle)'],
    'MCC': [0.65, 0.94, 0.9534],
    'Accuracy': [0.75, 0.97, 0.9709],
    'F1_Macro': [0.72, 0.96, 0.9675],
    'Params (M)': [0, 110, 0.74]  # Millions de paramètres
}

comp_df = pd.DataFrame(comparison_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Métriques de performance
ax1 = axes[0]
x = np.arange(len(comp_df['Model']))
width = 0.25

bars1 = ax1.bar(x - width, comp_df['MCC'], width, label='MCC', color='#3498db', alpha=0.8)
bars2 = ax1.bar(x, comp_df['Accuracy'], width, label='Accuracy', color='#2ecc71', alpha=0.8)
bars3 = ax1.bar(x + width, comp_df['F1_Macro'], width, label='F1 Macro', color='#e74c3c', alpha=0.8)

ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
ax1.set_title('Comparaison des Performances', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(comp_df['Model'])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0.6, 1.0)

# Annotations
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

# Subplot 2: Efficacité (MCC vs Params)
ax2 = axes[1]
colors_map = ['#95a5a6', '#e67e22', '#27ae60']
sizes = [100, 300, 200]

for i, (model, mcc, params, color, size) in enumerate(zip(
    comp_df['Model'], comp_df['MCC'], comp_df['Params (M)'], colors_map, sizes)):
    ax2.scatter(params, mcc, s=size, color=color, alpha=0.7, edgecolors='black', linewidth=2)
    ax2.annotate(model.replace('\n', ' '), 
                xy=(params, mcc), 
                xytext=(10, 10), 
                textcoords='offset points',
                fontsize=9, 
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))

ax2.set_xlabel('Paramètres Entraînables (Millions)', fontsize=12, fontweight='bold')
ax2.set_ylabel('MCC', fontsize=12, fontweight='bold')
ax2.set_title('Efficacité: MCC vs Paramètres', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-5, 120)

plt.tight_layout()
plt.savefig(figures_dir / 'comparison_sota.png', dpi=300, bbox_inches='tight')
print(f"    Sauvegardé: comparison_sota.png")
plt.close()

# ============================================================
# GRAPHIQUE 3: Matrices de confusion agrégées
# ============================================================
print(" Graphique 3: Matrices de confusion...")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Matrice de confusion moyenne
cm_sum = np.zeros((3, 3))

for idx, result in enumerate(results):
    cm = np.array(result['confusion_matrix'])
    cm_sum += cm
    
    # Normaliser par ligne (support)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Afficher
    ax = axes[idx]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', 
                xticklabels=['Neg', 'Neu', 'Pos'],
                yticklabels=['Neg', 'Neu', 'Pos'],
                ax=ax, cbar_kws={'label': 'Proportion'})
    ax.set_title(f'Fold {idx} (MCC: {result["mcc"]:.4f})', fontweight='bold')
    ax.set_ylabel('Vraie Classe', fontweight='bold')
    ax.set_xlabel('Classe Prédite', fontweight='bold')

# Matrice moyenne
cm_mean = cm_sum / len(results)
cm_mean_norm = cm_mean.astype('float') / cm_mean.sum(axis=1)[:, np.newaxis]

ax = axes[5]
sns.heatmap(cm_mean_norm, annot=True, fmt='.2%', cmap='RdYlGn', 
            xticklabels=['Neg', 'Neu', 'Pos'],
            yticklabels=['Neg', 'Neu', 'Pos'],
            ax=ax, cbar_kws={'label': 'Proportion'}, vmin=0, vmax=1)
ax.set_title(f'Moyenne des 5 Folds', fontweight='bold', fontsize=14)
ax.set_ylabel('Vraie Classe', fontweight='bold')
ax.set_xlabel('Classe Prédite', fontweight='bold')

plt.tight_layout()
plt.savefig(figures_dir / 'confusion_matrices.png', dpi=300, bbox_inches='tight')
print(f"   Sauvegardé: confusion_matrices.png")
plt.close()

# ============================================================
# GRAPHIQUE 4: Distribution des métriques
# ============================================================
print(" Graphique 4: Distribution des métriques...")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_data = {
    'Accuracy': results_df['accuracy'].values,
    'MCC': results_df['mcc'].values,
    'F1 Macro': results_df['f1_macro'].values
}

colors_box = ['#2ecc71', '#3498db', '#e74c3c']

for ax, (metric_name, values), color in zip(axes, metrics_data.items(), colors_box):
    # Boxplot
    bp = ax.boxplot([values], patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.7)
    
    # Points individuels
    ax.scatter(np.ones(len(values)), values, s=100, alpha=0.6, 
              color=color, edgecolors='black', linewidth=2, zorder=3)
    
    # Statistiques
    mean_val = values.mean()
    std_val = values.std()
    
    ax.axhline(mean_val, color='red', linestyle='--', linewidth=2, 
              label=f'μ = {mean_val:.4f}')
    ax.axhline(mean_val + std_val, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axhline(mean_val - std_val, color='orange', linestyle=':', linewidth=1.5, alpha=0.7,
              label=f'σ = {std_val:.4f}')
    
    ax.set_title(f'{metric_name}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12)
    ax.set_xticks([])
    ax.legend(loc='lower right')
    ax.grid(axis='y', alpha=0.3)
    
    # Annotations des valeurs
    for i, v in enumerate(values):
        ax.annotate(f'{v:.4f}', xy=(1, v), xytext=(1.15, v),
                   fontsize=8, ha='left', va='center')

plt.tight_layout()
plt.savefig(figures_dir / 'metrics_distribution.png', dpi=300, bbox_inches='tight')
print(f"    Sauvegardé: metrics_distribution.png")
plt.close()

# ============================================================
# RÉSUMÉ
# ============================================================
print(f"\n{'='*60}")
print(" TOUS LES GRAPHIQUES GÉNÉRÉS!")
print(f"{'='*60}\n")

print(f" Dossier: {figures_dir}\n")
print(f" Fichiers créés:")
print(f"   1. fold_performance.png")
print(f"   2. comparison_sota.png")
print(f"   3. confusion_matrices.png")
print(f"   4. metrics_distribution.png")

print(f"\n Utilisation dans le rapport:")
print(f"   • Figure 1: Section \"Résultats par Fold\"")
print(f"   • Figure 2: Section \"Comparaison État de l'Art\"")
print(f"   • Figure 3: Section \"Analyse des Erreurs\"")
print(f"   • Figure 4: Section \"Robustesse du Modèle\"")

print(f"\n{'='*60}")
print(f" PROCHAINE ÉTAPE: Rédaction du rapport")
print(f"{'='*60}")


 GÉNÉRATION DES GRAPHIQUES POUR LE RAPPORT

 Graphique 1: Performance par fold...
    Sauvegardé: fold_performance.png
 Graphique 2: Comparaison avec état de l'art...
    Sauvegardé: comparison_sota.png
 Graphique 3: Matrices de confusion...
   Sauvegardé: confusion_matrices.png
 Graphique 4: Distribution des métriques...
    Sauvegardé: metrics_distribution.png

 TOUS LES GRAPHIQUES GÉNÉRÉS!

 Dossier: c:\Users\lenovo\OneDrive\Bureau\cours\sentiTrade-HMA-V2\figures

 Fichiers créés:
   1. fold_performance.png
   2. comparison_sota.png
   3. confusion_matrices.png
   4. metrics_distribution.png

 Utilisation dans le rapport:
   • Figure 1: Section "Résultats par Fold"
   • Figure 2: Section "Comparaison État de l'Art"
   • Figure 3: Section "Analyse des Erreurs"
   • Figure 4: Section "Robustesse du Modèle"

 PROCHAINE ÉTAPE: Rédaction du rapport


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel, PeftConfig
import torch
import pandas as pd
import numpy as np
from datetime import datetime
import json

print(" PIPELINE D'INFÉRENCE - ANALYSE DE SENTIMENT")
print(f"{'='*60}\n")

# ============================================================
# 1. CHARGEMENT DU MEILLEUR MODÈLE 
# ============================================================
print("📥 Chargement du meilleur modèle...")

# Meilleur fold
results_path = project_root / "results" / "qlora_5fold_final_results.json"
with open(results_path, 'r') as f:
    results = json.load(f)

best_fold = max(results, key=lambda x: x['mcc'])
best_fold_idx = best_fold['fold']
best_mcc = best_fold['mcc']

print(f" Meilleur fold: {best_fold_idx} (MCC={best_mcc:.4f})\n")

# Charger configuration LoRA
model_path = project_root / "models" / "qlora_5fold" / f"fold_{best_fold_idx}_final"
peft_config = PeftConfig.from_pretrained(model_path)

# Charger modèle base avec 3 classes 
base_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,  
    id2label={0: 'negative', 1: 'neutral', 2: 'positive'},
    label2id={'negative': 0, 'neutral': 1, 'positive': 2}
)

# Charger les poids LoRA
model = PeftModel.from_pretrained(base_model, model_path)
model = model.merge_and_unload()  # Fusionner LoRA avec le modèle base
model.eval()

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

print(f" Modèle chargé et prêt!\n")

# ============================================================
# 2. FONCTION D'INFÉRENCE OPTIMISÉE
# ============================================================
def predict_sentiment_batch(texts, batch_size=32):
    """Prédiction de sentiment optimisée par batch"""
    all_sentiments = []
    all_probs = []
    all_scores = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        
        # Convertir prédictions
        id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
        sentiments = [id2label[p] for p in preds]
        
        # Score normalisé [-1, 1]: P(pos) - P(neg)
        scores = probs[:, 2] - probs[:, 0]
        
        all_sentiments.extend(sentiments)
        all_probs.extend(probs)
        all_scores.extend(scores)
    
    return {
        'sentiments': all_sentiments,
        'probabilities': np.array(all_probs),
        'scores': np.array(all_scores)
    }

print(f" Pipeline créé\n")

# ============================================================
# 3. TEST SUR DONNÉES FINANCIÈRES
# ============================================================
print(f"{'='*60}")
print(" TEST D'INFÉRENCE")
print(f"{'='*60}\n")

test_samples = [
    "Bitcoin surged 15% after strong institutional buying",
    "Company announces bankruptcy and layoff of 5000 employees",
    "Quarterly earnings remain unchanged from last period",
    "Stock hits all-time high on record revenue",
    "Market crash wipes out $2 trillion in value",
    "Dividend payout scheduled for next month",
    "CEO resigns amid fraud investigation",
    "Revenue growth exceeds analyst forecasts by 30%"
]

results = predict_sentiment_batch(test_samples)

# Affichage formaté
print(f"{'─'*80}")
print(f"{'Texte':<55} {'Sentiment':<12} {'Score':>8}")
print(f"{'─'*80}")

for text, sentiment, score, probs in zip(
    test_samples, 
    results['sentiments'], 
    results['scores'],
    results['probabilities']
):
    emoji = {'positive': '🟢', 'neutral': '🟡', 'negative': '🔴'}[sentiment]
    text_short = text[:52] + "..." if len(text) > 55 else text
    conf = probs.max()
    
    print(f"{text_short:<55} {emoji} {sentiment:<10} {score:>7.3f} ({conf:.2%})")

print(f"{'─'*80}\n")

# Statistiques
sentiment_dist = pd.Series(results['sentiments']).value_counts()
print(f" Distribution:")
for sent, count in sentiment_dist.items():
    print(f"   {sent.capitalize()}: {count}/8 ({count/8*100:.0f}%)")

print(f"\n Modèle opérationnel!\n")

# ============================================================
# 4. SAUVEGARDER LE PIPELINE
# ============================================================
print(f" Sauvegarde du pipeline...")

# Sauvegarder modèle fusionné
production_model_path = project_root / "models" / "sentiment_production"
production_model_path.mkdir(exist_ok=True, parents=True)

model.save_pretrained(production_model_path)
tokenizer.save_pretrained(production_model_path)

# Sauvegarder métadonnées
metadata = {
    'model': 'distilbert-base-uncased',
    'best_fold': best_fold_idx,
    'mcc': best_mcc,
    'accuracy': best_fold['accuracy'],
    'num_labels': 3,
    'labels': ['negative', 'neutral', 'positive'],
    'training_date': datetime.now().isoformat()
}

with open(production_model_path / "metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f" Pipeline sauvegardé: {production_model_path}\n")

print(f"{'='*60}")
print(f" PIPELINE OPÉRATIONNEL!")
print(f"{'='*60}")


 PIPELINE D'INFÉRENCE - ANALYSE DE SENTIMENT

📥 Chargement du meilleur modèle...
 Meilleur fold: 0 (MCC=0.9599)



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Modèle chargé et prêt!

 Pipeline créé

 TEST D'INFÉRENCE

────────────────────────────────────────────────────────────────────────────────
Texte                                                   Sentiment       Score
────────────────────────────────────────────────────────────────────────────────
Bitcoin surged 15% after strong institutional buying    🟢 positive     0.999 (99.96%)
Company announces bankruptcy and layoff of 5000 empl... 🟡 neutral     -0.001 (99.90%)
Quarterly earnings remain unchanged from last period    🟡 neutral      0.042 (92.84%)
Stock hits all-time high on record revenue              🟢 positive     0.990 (99.20%)
Market crash wipes out $2 trillion in value             🔴 negative    -0.990 (99.11%)
Dividend payout scheduled for next month                🟡 neutral     -0.000 (99.99%)
CEO resigns amid fraud investigation                    🟡 neutral     -0.001 (99.86%)
Revenue growth exceeds analyst forecasts by 30%         🟢 positive     0.294 (63.70%)
────────────

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm

print(" COLLECTE DONNÉES FINANCIÈRES + SENTIMENT")
print(f"{'='*60}\n")

# ============================================================
# 1. TÉLÉCHARGER DONNÉES DE PRIX
# ============================================================
print(" Téléchargement données de prix...")

# Actifs à trader
SYMBOLS = ['BTC-USD', 'ETH-USD']  # Cryptos pour début
END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=365)

print(f"   Période: {START_DATE.date()} → {END_DATE.date()}")
print(f"   Symboles: {', '.join(SYMBOLS)}\n")

market_data = {}

for symbol in SYMBOLS:
    print(f"   Téléchargement {symbol}...", end=" ")
    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(start=START_DATE, end=END_DATE, interval='1h')
        
        if len(df) > 0:
            market_data[symbol] = df
            print(f" {len(df):,} heures")
        else:
            print(f" Aucune donnée")
    except Exception as e:
        print(f"❌ {e}")

print(f"\n {len(market_data)} actifs collectés\n")

# ============================================================
# 2. CALCULER INDICATEURS TECHNIQUES
# ============================================================
print("🔧 Calcul indicateurs techniques...")

def add_technical_indicators(df):
    """Ajoute indicateurs techniques au DataFrame"""
    df = df.copy()
    
    # Returns
    df['returns'] = df['Close'].pct_change()
    df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # Moving Averages
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()
    df['EMA_12'] = df['Close'].ewm(span=12).mean()
    df['EMA_26'] = df['Close'].ewm(span=26).mean()
    
    # MACD
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9).mean()
    df['MACD_hist'] = df['MACD'] - df['MACD_signal']
    
    # RSI
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Bollinger Bands
    df['BB_middle'] = df['Close'].rolling(20).mean()
    df['BB_std'] = df['Close'].rolling(20).std()
    df['BB_upper'] = df['BB_middle'] + 2 * df['BB_std']
    df['BB_lower'] = df['BB_middle'] - 2 * df['BB_std']
    df['BB_width'] = (df['BB_upper'] - df['BB_lower']) / df['BB_middle']
    
    # Volatility
    df['volatility'] = df['returns'].rolling(24).std() * np.sqrt(24)  # Annualisée
    
    # Volume indicators
    df['volume_sma'] = df['Volume'].rolling(20).mean()
    df['volume_ratio'] = df['Volume'] / df['volume_sma']
    
    # Price momentum
    df['momentum_1h'] = df['Close'].pct_change(1)
    df['momentum_4h'] = df['Close'].pct_change(4)
    df['momentum_24h'] = df['Close'].pct_change(24)
    
    return df

for symbol in market_data.keys():
    market_data[symbol] = add_technical_indicators(market_data[symbol])
    print(f"    {symbol}: {len(market_data[symbol].columns)} features")

print(f"\n Indicateurs calculés\n")

# ============================================================
# 3. SIMULER DONNÉES NEWS/SENTIMENT (À REMPLACER PAR VRAIES NEWS)
# ============================================================
print(" Génération features sentiment...")

def generate_mock_sentiment_data(df, symbol):
    """
    TEMPORAIRE: Génère des données sentiment factices
    À REMPLACER par vrai scraping news + prédiction
    """
    np.random.seed(42)
    
    # Simuler sentiment corrélé avec returns
    sentiment_scores = []
    
    for i, row in df.iterrows():
        if pd.isna(row['returns']):
            score = 0.0
        else:
            # Sentiment suit les returns avec du bruit
            base_sentiment = np.tanh(row['returns'] * 50)  # Tanh pour [-1, 1]
            noise = np.random.normal(0, 0.2)
            score = np.clip(base_sentiment + noise, -1, 1)
        
        sentiment_scores.append(score)
    
    df['sentiment_score'] = sentiment_scores
    
    # Features sentiment dérivées
    df['sentiment_ma_4h'] = df['sentiment_score'].rolling(4).mean()
    df['sentiment_ma_24h'] = df['sentiment_score'].rolling(24).mean()
    df['sentiment_volatility'] = df['sentiment_score'].rolling(24).std()
    df['sentiment_momentum'] = df['sentiment_score'].diff()
    
    # Sentiment catégoriel
    df['sentiment'] = pd.cut(
        df['sentiment_score'],
        bins=[-1, -0.3, 0.3, 1],
        labels=['negative', 'neutral', 'positive']
    )
    
    return df

for symbol in market_data.keys():
    market_data[symbol] = generate_mock_sentiment_data(market_data[symbol], symbol)
    print(f"    {symbol}: sentiment features ajoutées")

print(f"\n  NOTE: Données sentiment simulées")
print(f"   → Prochaine étape: Intégrer vrai scraping + prédiction\n")

# ============================================================
# 4. PRÉPARER TARGET (PRIX FUTURS)
# ============================================================
print(" Génération targets pour TFT...")

def add_targets(df, horizons=[1, 4, 24]):
    """Ajoute prix futurs comme targets"""
    df = df.copy()
    
    for h in horizons:
        # Prix futur
        df[f'target_price_{h}h'] = df['Close'].shift(-h)
        
        # Returns futurs
        df[f'target_return_{h}h'] = df[f'target_price_{h}h'] / df['Close'] - 1
        
        # Direction (classification binaire)
        df[f'target_direction_{h}h'] = (df[f'target_return_{h}h'] > 0).astype(int)
    
    return df

for symbol in market_data.keys():
    market_data[symbol] = add_targets(market_data[symbol])
    print(f"    {symbol}: targets créés (1h, 4h, 24h)")

print(f"\n Targets générés\n")

# ============================================================
# 5. NETTOYER ET SAUVEGARDER
# ============================================================
print(" Nettoyage final...")

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(exist_ok=True, parents=True)

for symbol, df in market_data.items():
    # Supprimer NaN (dus au rolling)
    df_clean = df.dropna()
    
    # Sauvegarder
    filename = processed_dir / f"{symbol.replace('-', '_')}_1h_with_sentiment.csv"
    df_clean.to_csv(filename)
    
    print(f"    {symbol}: {len(df_clean):,} lignes → {filename.name}")

print(f"\n{'='*60}")
print(f" DONNÉES PRÊTES POUR LE TFT!")
print(f"{'='*60}\n")

# Statistiques finales
print(f" Résumé:")
for symbol, df in market_data.items():
    df_clean = df.dropna()
    print(f"\n{symbol}:")
    print(f"   Période: {df_clean.index[0]} → {df_clean.index[-1]}")
    print(f"   Points: {len(df_clean):,}")
    print(f"   Features: {len(df_clean.columns)}")
    print(f"   Sentiment moyen: {df_clean['sentiment_score'].mean():.3f}")
    print(f"   Returns moyens: {df_clean['returns'].mean()*100:.4f}%/h")

print(f"\n PROCHAINE ÉTAPE:")
print(f"   → Entraîner le Temporal Fusion Transformer")
print(f"   → Prédiction multi-horizon (1h, 4h, 24h)")


 COLLECTE DONNÉES FINANCIÈRES + SENTIMENT

 Téléchargement données de prix...
   Période: 2025-01-02 → 2026-01-02
   Symboles: BTC-USD, ETH-USD

   Téléchargement BTC-USD...  8,604 heures
   Téléchargement ETH-USD...  8,604 heures

 2 actifs collectés

🔧 Calcul indicateurs techniques...
    BTC-USD: 28 features
    ETH-USD: 28 features

 Indicateurs calculés

 Génération features sentiment...
    BTC-USD: sentiment features ajoutées
    ETH-USD: sentiment features ajoutées

  NOTE: Données sentiment simulées
   → Prochaine étape: Intégrer vrai scraping + prédiction

 Génération targets pour TFT...
    BTC-USD: targets créés (1h, 4h, 24h)
    ETH-USD: targets créés (1h, 4h, 24h)

 Targets générés

 Nettoyage final...
    BTC-USD: 8,425 lignes → BTC_USD_1h_with_sentiment.csv
    ETH-USD: 8,450 lignes → ETH_USD_1h_with_sentiment.csv

 DONNÉES PRÊTES POUR LE TFT!

 Résumé:

BTC-USD:
   Période: 2025-01-04 16:00:00+00:00 → 2026-01-01 13:00:00+00:00
   Points: 8,425
   Features: 43
   Sentim